# 1. PINN · 방정식과 경계조건으로 해 구하기

구간 $0\le x\le 1$에서 다음 문제를 풉니다.

$$u''(x)=1,\qquad u(0)=u(1)=0.$$

신경망은 좌표 $x$를 받아 $u(x)$를 출력합니다. 방정식 잔차와 경계조건 오차로 학습합니다. 비교에 사용할 해석해는 $u(x)=x(x-1)/2$이며, 학습 target으로 넣지 않습니다.

이번 실습에서는 **준비된 PINN을 실행하고, AI에 요청한 그래프 코드로 실제 결과를 그립니다.**


## 1. 준비


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pinn_basics.py").is_file()
             and (p / "02_FNO.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("워크숍 교재 폴더 안에서 이 노트북을 여세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("교재:", ROOT)
print("Python:", sys.executable)


In [ ]:
import json
import os
import subprocess
from uuid import uuid4
import numpy as np
import torch
from IPython.display import display, Markdown

if not torch.cuda.is_available():
    raise RuntimeError("00_Start_Here의 GPU 확인부터 진행하세요.")
DEVICE = "cuda"
PINN_COMPLETED = False


## 2. 준비된 모델 학습

아래 셀은 원본과 같은 seed 42, FP32, 1,000회 L-BFGS 호출을 실행합니다. L-BFGS 한 호출에서 loss가 여러 번 계산될 수 있습니다. 실행이 끝날 때까지 기다립니다.


In [ ]:
PINN_COMPLETED = False
PINN_RUN = ROOT / "runs" / f"pinn-{uuid4().hex[:10]}"
command = [sys.executable, "-m", "src.pinn_basics", "--mode", "forward",
           "--device", DEVICE, "--steps", "1000", "--seed", "42",
           "--output-dir", str(PINN_RUN)]
env = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
subprocess.run(command, cwd=ROOT, env=env, check=True)
PINN_COMPLETED = True
print("결과:", PINN_RUN.relative_to(ROOT))


## 3. 실제 결과 배열 읽기


In [ ]:
if not PINN_COMPLETED:
    raise RuntimeError("바로 위 학습 셀을 완료하세요.")
with np.load(PINN_RUN / "predictions.npz", allow_pickle=False) as saved:
    x = saved["x"].reshape(-1)
    u_pred = saved["prediction"].reshape(-1)
    u_exact = saved["reference"].reshape(-1)
metrics = json.loads((PINN_RUN / "metrics.json").read_text())
print("배열 shape:", x.shape, u_pred.shape, u_exact.shape)
print(json.dumps(metrics["heldout_after"], indent=2))
print("이 예제의 정확도 기준 충족:", metrics["accuracy"]["passed"])


## 4. AI에 그래프 코드 요청

아래 문장을 Gemini 또는 ChatGPT에 그대로 복사합니다. 반환된 Python 코드만 다음 셀에 붙여 넣고 실행하세요.

> Jupyter에서 PINN을 실행했어. `x`, `u_pred`, `u_exact`는 이미 계산된 길이 401의 1차원 NumPy 배열이야. 이 배열을 그대로 사용해서 matplotlib 그래프 2개를 나란히 그리는 코드만 작성해 줘. 왼쪽에는 `u_pred`와 `u_exact`를 겹쳐 그리고 범례를 넣어 줘. 오른쪽에는 `u_pred - u_exact`와 0 기준선을 그려 줘. 축 이름과 제목은 영어로 해 줘. 데이터를 새로 만들거나 배열을 수정하지 말고, 모델 학습이나 결과 해석은 하지 마.


In [ ]:
# AI가 작성한 그래프 코드만 이 셀에 붙여 넣고 실행하세요.
# x, u_pred, u_exact는 위 셀에서 이미 준비했습니다.


## 5. 강사와 함께 확인

예측과 해석해의 모양은 같은가요? 두 곡선이 겹쳐 보여도 오차 그래프가 필요한 이유는 무엇인가요? 방정식 잔차와 경계 오차는 각각 무엇을 확인하나요?

다음은 [02_FNO.ipynb](02_FNO.ipynb)입니다.

## 선택: 그래프 작성이 막힌 경우

강사 안내에 따라 아래 두 줄의 주석을 풀고 실행할 수 있습니다. 자동으로 실행되는 정답 기능은 아닙니다.


In [ ]:
# from instructor.pinn_plot import plot_pinn
# display(plot_pinn(x, u_pred, u_exact))
